# Rainfall Trigger-Threshold Model

This notebook complements `landslide_susceptibility_modeling.ipynb` (the **where**
model) with a rainfall-only **when** model, using antecedent-rainfall data
(IMERG, 1h -> 30-day accumulations) for the same 6,180 landslide events and
6,180 paired non-landslide (pseudo-absence) points.

**Read this before the results below.** Each non-landslide point is paired with one
landslide: it lies 2-10 km away (up to ~22 km where the ring was full), in the same
province, at least 2 km from every landslide record, and it **takes its paired
landslide's date** (`scripts/generate_paired_negatives.py`). Pairs therefore share
the same weather system, and IMERG's grid cells are ~10 km, so a pair's rainfall is
nearly identical **by design**. This dataset can only show whether small
within-pair differences in rainfall carry any signal; it cannot support a
rainfall-amount threshold, and an AUC near 0.5 is the expected, honest result.

Also note that the landslide "dates" are satellite-image dates of scar-mapping layers,
not occurrence dates, and ~16 % of them may have day and month swapped (see
`docs/METHODS.md`, Known issues 3). Treat everything below as an exploratory pattern
analysis, not a calibrated threshold.


In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import loguniform, randint
from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, classification_report, cohen_kappa_score, confusion_matrix,
    ConfusionMatrixDisplay, f1_score, precision_recall_curve, precision_score,
    recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import joblib

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')

RANDOM_STATE = 42
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data/processed/landslide_event_antecedent_rainfall.csv').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
ARTIFACT_DIR = PROJECT_ROOT / 'models/rainfall_trigger'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load and Combine

In [2]:
landslide = pd.read_csv(PROJECT_ROOT / 'data/processed/landslide_event_antecedent_rainfall.csv')
non_landslide = pd.read_csv(PROJECT_ROOT / 'data/processed/non_landslide_event_antecedent_rainfall.csv')
landslide['landslide'] = 1
non_landslide['landslide'] = 0
assert list(landslide.columns) == list(non_landslide.columns)
df = pd.concat([landslide, non_landslide], ignore_index=True)
print('Shape:', df.shape)
print(df['rainfall_status'].value_counts())
df['landslide'].value_counts()

Shape: (12360, 83)
rainfall_status
complete    12360
Name: count, dtype: int64


landslide
1    6180
0    6180
Name: count, dtype: int64

In [3]:
# Every window's mm total and coverage pct -- coverage should be ~100% (full IMERG record).
mm_cols = [c for c in df.columns if c.endswith('_mm')]
cov_cols = [c for c in df.columns if c.endswith('_coverage_pct')]
print('Missing values in mm columns:', int(df[mm_cols].isna().sum().sum()))
print('Min coverage pct:', df[cov_cols].min().min())

Missing values in mm columns: 0
Min coverage pct: 100.0


## 2. EDA: The Paired-Sampling Effect

As flagged above, each non-landslide point shares its paired landslide's date and
region, so the two classes see almost the same rainfall. The table and plots below
make that explicit.


In [4]:
key_windows = ['rain_1h_mm', 'rain_3h_mm', 'rain_6h_mm', 'rain_1d_mm', 'rain_3d_mm',
               'rain_7d_mm', 'rain_15d_mm', 'rain_30d_mm']
df.groupby('landslide')[key_windows].agg(['mean', 'median']).T

landslide                   0          1
rain_1h_mm  mean     0.002985   0.001812
            median   0.000000   0.000000
rain_3h_mm  mean     0.007011   0.006484
            median   0.000000   0.000000
rain_6h_mm  mean     0.051439   0.056658
            median   0.000000   0.000000
rain_1d_mm  mean     1.410947   1.265272
            median   0.000000   0.170000
rain_3d_mm  mean     7.709623   9.291283
            median   3.535000   3.628000
rain_7d_mm  mean    17.286579  18.768969
            median  16.755000  20.094499
rain_15d_mm mean    38.558010  39.684387
            median  33.844999  33.936499
rain_30d_mm mean    66.973951  67.091141
            median  58.074998  54.986499

In [5]:
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for ax, col in zip(axes.ravel(), key_windows):
    sns.boxplot(data=df, x='landslide', y=col, showfliers=False,
                palette=['#F58518', '#4C78A8'], ax=ax)
    ax.set_title(col)
    ax.set_xticklabels(['Non-landslide\n(paired, same date)', 'Landslide'])
plt.suptitle('Rainfall by class: each non-landslide point shares the date and region of its paired landslide', y=1.02)
plt.tight_layout()
plt.show()

## 3. Skewness and Correlation

Nested accumulation windows (e.g. `rain_7d_mm` includes `rain_1d_mm`) are
strongly correlated by construction; this motivates the engineered,
less-redundant feature set in section 5 rather than using all 33 raw windows.

In [6]:
skew = df[mm_cols].skew().sort_values(key=lambda s: s.abs(), ascending=False)
print('Most skewed windows:')
print(skew.head(8))
plt.figure(figsize=(14, 11))
corr = df[key_windows + ['rain_10d_mm', 'rain_20d_mm']].corr(method='spearman')
sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag', center=0, vmin=-1, vmax=1, square=True)
plt.title('Spearman correlation across representative windows')
plt.tight_layout()
plt.show()

Most skewed windows:
rain_3h_mm     31.642979
rain_1h_mm     25.498443
rain_6h_mm     13.730839
rain_5d_mm      4.257997
rain_6d_mm      4.234724
rain_7d_mm      3.912337
rain_10d_mm     3.819329
rain_9d_mm      3.558845
dtype: float64


## 4. Classic Intensity-Duration View

The standard landslide-rainfall-threshold plot: mean intensity (mm/day) vs.
duration, log-log, landslide vs. non-landslide. Given the paired same-date sampling,
expect heavy overlap -- this is the honest result, not a bug.

In [7]:
id_durations = [1, 2, 3, 5, 7, 10, 15, 20, 25, 30]
id_rows = []
for d in id_durations:
    col = f'rain_{d}d_mm'
    for label, sub in df.groupby('landslide'):
        id_rows.append({'duration_days': d, 'landslide': label,
                         'intensity_mm_per_day': (sub[col] / d).replace(0, np.nan)})
id_long = pd.concat([
    pd.DataFrame({'duration_days': r['duration_days'], 'landslide': r['landslide'],
                  'intensity_mm_per_day': r['intensity_mm_per_day']})
    for r in id_rows
], ignore_index=True)

plt.figure(figsize=(8, 6))
sns.scatterplot(data=id_long.sample(min(6000, len(id_long)), random_state=RANDOM_STATE),
                x='duration_days', y='intensity_mm_per_day', hue='landslide',
                palette={0: '#F58518', 1: '#4C78A8'}, s=12, alpha=0.35, linewidth=0)
plt.xscale('log'); plt.yscale('log')
plt.xlabel('Duration (days, log scale)')
plt.ylabel('Mean intensity (mm/day, log scale)')
plt.title('Intensity-Duration: heavy overlap is expected (pairs share date and region)')
plt.tight_layout()
plt.show()

## 5. Feature Engineering

Rather than feed all 33 collinear raw windows into a linear model, build a
compact set that can capture *shape* differences the paired sampling does not
control for: short-burst intensity, short/medium/long accumulation, and
burst-concentration ratios (short accumulation relative to long antecedent).

In [8]:
def build_features(data):
    out = pd.DataFrame(index=data.index)
    out['burst_1h'] = data['rain_1h_mm']
    out['burst_3h'] = data['rain_3h_mm']
    out['burst_6h'] = data['rain_6h_mm']
    out['accum_1d'] = data['rain_1d_mm']
    out['accum_3d'] = data['rain_3d_mm']
    out['accum_7d'] = data['rain_7d_mm']
    out['accum_15d'] = data['rain_15d_mm']
    out['accum_30d'] = data['rain_30d_mm']
    eps = 1.0  # avoid divide-by-zero while keeping the ratio's scale interpretable
    out['concentration_1d_over_7d'] = data['rain_1d_mm'] / (data['rain_7d_mm'] + eps)
    out['concentration_3d_over_30d'] = data['rain_3d_mm'] / (data['rain_30d_mm'] + eps)
    out['concentration_7d_over_30d'] = data['rain_7d_mm'] / (data['rain_30d_mm'] + eps)
    out['burst_over_accum1d'] = data['rain_1h_mm'] / (data['rain_1d_mm'] + eps)
    out['antecedent_15_30d'] = data['rain_30d_mm'] - data['rain_15d_mm']
    return out

X = build_features(df)
y = df['landslide'].astype(int)
feature_cols = X.columns.tolist()
print('Features:', feature_cols)
X.describe().T[['mean', 'std', 'min', 'max']]

Features: ['burst_1h', 'burst_3h', 'burst_6h', 'accum_1d', 'accum_3d', 'accum_7d', 'accum_15d', 'accum_30d', 'concentration_1d_over_7d', 'concentration_3d_over_30d', 'concentration_7d_over_30d', 'burst_over_accum1d', 'antecedent_15_30d']


,mean,std,min,max
burst_1h,0.002398,0.045825,0.0,1.695000
burst_3h,0.006747,0.107485,0.0,4.855000
burst_6h,0.054048,0.391939,0.0,9.610000
accum_1d,1.338110,3.065621,0.0,34.899999
accum_3d,8.500453,10.806329,0.0,124.529997
accum_7d,18.027774,17.631241,0.0,291.789993
accum_15d,39.121199,37.078758,0.0,536.459989
accum_30d,67.032546,68.216965,0.0,719.634985
concentration_1d_over_7d,0.051319,0.114549,0.0,0.935858
concentration_3d_over_30d,0.152652,0.242290,0.0,0.869979


## 6. Spatially Blocked, Nested Cross-Validation

Same methodology as the susceptibility notebook: ~0.25 deg spatial blocks as
CV groups (`StratifiedGroupKFold`), so nearby/same-storm points never split
across train and test; nested CV (outer selection, inner hyperparameter
search) so the held-out test set is touched exactly once.

In [9]:
BLOCK_DEG = 0.25
N_OUTER, N_INNER = 5, 3
N_ITER = 8

groups = (np.floor(df['longitude'] / BLOCK_DEG).astype(int).astype(str) + '_' +
          np.floor(df['latitude'] / BLOCK_DEG).astype(int).astype(str))

outer_cv = StratifiedGroupKFold(n_splits=N_OUTER, shuffle=True, random_state=RANDOM_STATE)
inner_cv = StratifiedGroupKFold(n_splits=N_INNER, shuffle=True, random_state=RANDOM_STATE)

train_idx, test_idx = next(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
                           .split(X, y, groups))
X_train, y_train, g_train = X.iloc[train_idx], y.iloc[train_idx], groups.iloc[train_idx]
X_test, y_test, g_test = X.iloc[test_idx], y.iloc[test_idx], groups.iloc[test_idx]
assert not set(g_train) & set(g_test)
print('Train:', X_train.shape, y_train.value_counts().to_dict())
print('Test: ', X_test.shape, y_test.value_counts().to_dict())

Train: (9902, 13) {0: 5042, 1: 4860}
Test:  (2458, 13) {1: 1320, 0: 1138}


In [10]:
def make_models():
    m = {}
    m['logistic_regression'] = (
        Pipeline([('impute', SimpleImputer(strategy='median')),
                  ('scale', StandardScaler()),
                  ('model', LogisticRegression(max_iter=5000, class_weight='balanced', random_state=RANDOM_STATE))]),
        {'model__C': loguniform(1e-2, 1e2)})
    m['random_forest'] = (
        Pipeline([('impute', SimpleImputer(strategy='median')),
                  ('model', RandomForestClassifier(n_estimators=300, class_weight='balanced', n_jobs=1, random_state=RANDOM_STATE))]),
        {'model__max_depth': [4, 6, 10, None], 'model__min_samples_leaf': randint(2, 20)})
    m['gradient_boosting'] = (
        Pipeline([('impute', SimpleImputer(strategy='median')),
                  ('model', HistGradientBoostingClassifier(class_weight='balanced', random_state=RANDOM_STATE))]),
        {'model__learning_rate': loguniform(0.02, 0.2), 'model__max_depth': randint(2, 6),
         'model__max_iter': randint(80, 300), 'model__min_samples_leaf': randint(20, 100)})
    try:
        from xgboost import XGBClassifier
        pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())
        m['xgboost'] = (
            Pipeline([('impute', SimpleImputer(strategy='median')),
                      ('model', XGBClassifier(eval_metric='logloss', scale_pos_weight=pos_weight,
                                              random_state=RANDOM_STATE, n_jobs=1))]),
            {'model__n_estimators': randint(80, 400), 'model__learning_rate': loguniform(0.02, 0.2),
             'model__max_depth': randint(2, 6), 'model__subsample': [0.7, 0.85, 1.0]})
    except Exception as exc:
        print('Skipping XGBoost:', exc)
    try:
        from lightgbm import LGBMClassifier
        m['lightgbm'] = (
            Pipeline([('impute', SimpleImputer(strategy='median')),
                      ('model', LGBMClassifier(class_weight='balanced', random_state=RANDOM_STATE, n_jobs=1, verbose=-1))]),
            {'model__n_estimators': randint(80, 400), 'model__learning_rate': loguniform(0.02, 0.2),
             'model__num_leaves': randint(8, 48), 'model__min_child_samples': randint(10, 60)})
    except Exception as exc:
        print('Skipping LightGBM:', exc)
    return m

model_specs = make_models()
list(model_specs)

['logistic_regression',
 'random_forest',
 'gradient_boosting',
 'xgboost',
 'lightgbm']

In [11]:
def predict_proba_positive(model, X_data):
    return model.predict_proba(X_data)[:, 1]


def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return tn / (tn + fp)


def evaluate_predictions(y_true, proba, threshold=0.5):
    y_true = np.asarray(y_true)
    pred = (proba >= threshold).astype(int)
    return {
        'accuracy': accuracy_score(y_true, pred),
        'precision': precision_score(y_true, pred, zero_division=0),
        'recall_sensitivity': recall_score(y_true, pred, zero_division=0),
        'specificity': specificity_score(y_true, pred),
        'f1': f1_score(y_true, pred, zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(y_true, pred),
        'roc_auc': roc_auc_score(y_true, proba),
        'pr_auc': average_precision_score(y_true, proba),
        'brier': brier_score_loss(y_true, proba),
        'cohen_kappa': cohen_kappa_score(y_true, pred),
    }


def youden_threshold(y_true, proba):
    fpr, tpr, thr = roc_curve(y_true, proba)
    return float(thr[np.argmax(tpr - fpr)])


def make_search(pipe, dist, n_iter=N_ITER):
    return RandomizedSearchCV(pipe, dist, n_iter=n_iter, scoring='roc_auc', cv=inner_cv,
                              n_jobs=-1, random_state=RANDOM_STATE, refit=True)

In [12]:
import time

oof_proba = {name: np.full(len(X_train), np.nan) for name in model_specs}
fold_rows = []
for name, (pipe, dist) in model_specs.items():
    t0 = time.time()
    for fold, (tr, va) in enumerate(outer_cv.split(X_train, y_train, g_train)):
        search = make_search(clone(pipe), dist)
        search.fit(X_train.iloc[tr], y_train.iloc[tr], groups=g_train.iloc[tr])
        p = predict_proba_positive(search.best_estimator_, X_train.iloc[va])
        oof_proba[name][va] = p
        fold_rows.append({'model': name, 'fold': fold, **evaluate_predictions(y_train.iloc[va], p, 0.5)})
    print(f'{name}: nested CV done in {time.time() - t0:.0f}s')

fold_df = pd.DataFrame(fold_rows)
metric_cols_cv = ['accuracy', 'precision', 'recall_sensitivity', 'specificity', 'f1', 'balanced_accuracy', 'roc_auc', 'pr_auc', 'brier']
agg = fold_df.groupby('model')[metric_cols_cv].agg(['mean', 'std'])
agg.columns = [f'{a}_{b}' for a, b in agg.columns]
cv_results = agg.sort_values('roc_auc_mean', ascending=False).reset_index()
cv_results

logistic_regression: nested CV done in 2s


random_forest: nested CV done in 11s


gradient_boosting: nested CV done in 7s


xgboost: nested CV done in 2s


lightgbm: nested CV done in 2s


,model,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_sensitivity_mean,recall_sensitivity_std,specificity_mean,specificity_std,f1_mean,f1_std,balanced_accuracy_mean,balanced_accuracy_std,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,brier_mean,brier_std
0,random_forest,0.601966,0.123644,0.683792,0.134346,0.326460,0.190017,0.845778,0.161312,0.422465,0.197348,0.586119,0.106264,0.736545,0.099522,0.758697,0.067335,0.219056,0.054047
1,gradient_boosting,0.584666,0.145568,0.629500,0.168930,0.324976,0.177153,0.821108,0.117733,0.420594,0.183894,0.573042,0.120920,0.715730,0.122724,0.732596,0.086408,0.246446,0.070502
2,lightgbm,0.608659,0.127156,0.673994,0.172731,0.332709,0.198575,0.875746,0.047980,0.431048,0.208317,0.604227,0.108008,0.706794,0.204699,0.737043,0.137690,0.269566,0.130899
3,xgboost,0.627785,0.129349,0.693702,0.159899,0.403717,0.204853,0.857200,0.047049,0.493460,0.199983,0.630459,0.106062,0.686500,0.156101,0.731048,0.093077,0.252776,0.095016
4,logistic_regression,0.409825,0.087083,0.372858,0.072868,0.367582,0.188161,0.473924,0.049064,0.358478,0.128052,0.420753,0.083549,0.457579,0.068441,0.529602,0.025965,0.305322,0.085636


In [13]:
cv_plot = cv_results.set_index('model')[['roc_auc_mean', 'pr_auc_mean', 'f1_mean', 'balanced_accuracy_mean']].sort_values('roc_auc_mean')
cv_plot.plot(kind='barh', figsize=(9, 4))
plt.axvline(0.5, color='red', linestyle='--', linewidth=1, label='no-skill (AUC=0.5)')
plt.xlim(0, 1)
plt.xlabel('Nested spatial CV score')
plt.title('Rainfall-only trigger model comparison')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Final Fit and Held-out Evaluation

In [14]:
best_name = cv_results.iloc[0]['model']
oof_threshold = {n: youden_threshold(y_train, p) for n, p in oof_proba.items()}

fitted_models, best_params, test_proba = {}, {}, {}
test_rows = []
rng = np.random.default_rng(RANDOM_STATE)
boot_idx = [rng.integers(0, len(y_test), len(y_test)) for _ in range(300)]
y_test_arr = y_test.to_numpy()

for name, (pipe, dist) in model_specs.items():
    search = make_search(clone(pipe), dist)
    search.fit(X_train, y_train, groups=g_train)
    fitted_models[name] = search.best_estimator_
    best_params[name] = {k: (v.item() if hasattr(v, 'item') else v) for k, v in search.best_params_.items()}
    proba = predict_proba_positive(search.best_estimator_, X_test)
    test_proba[name] = proba
    aucs = [roc_auc_score(y_test_arr[i], proba[i]) for i in boot_idx if len(np.unique(y_test_arr[i])) == 2]
    row = {'model': name, 'threshold': oof_threshold[name]}
    row.update(evaluate_predictions(y_test, proba, threshold=oof_threshold[name]))
    row['roc_auc_ci_low'], row['roc_auc_ci_high'] = np.percentile(aucs, [2.5, 97.5])
    test_rows.append(row)

test_results = pd.DataFrame(test_rows).sort_values('roc_auc', ascending=False).reset_index(drop=True)
print('Selected by nested CV:', best_name)
test_results

Selected by nested CV: random_forest


,model,threshold,accuracy,precision,recall_sensitivity,specificity,f1,balanced_accuracy,roc_auc,pr_auc,brier,cohen_kappa,roc_auc_ci_low,roc_auc_ci_high
0,gradient_boosting,0.130981,0.666395,0.625755,0.942424,0.346221,0.752116,0.644323,0.536990,0.552941,0.320376,0.300819,0.513601,0.560701
1,logistic_regression,0.650073,0.462571,0.473684,0.006818,0.991213,0.013443,0.499015,0.417828,0.532753,0.259123,-0.001825,0.392650,0.441387
2,xgboost,0.103160,0.558584,0.549225,0.993182,0.054482,0.707310,0.523832,0.347021,0.462604,0.351399,0.050957,0.322179,0.369396
3,random_forest,0.293839,0.347437,0.373665,0.318182,0.381371,0.343699,0.349776,0.338080,0.463578,0.340201,-0.296922,0.317618,0.361071
4,lightgbm,0.473723,0.322213,0.127155,0.044697,0.644112,0.066143,0.344405,0.311102,0.454569,0.428387,-0.295849,0.292374,0.335957


In [15]:
best_model = fitted_models[best_name]
best_proba = test_proba[best_name]
best_threshold = oof_threshold[best_name]
best_pred = (best_proba >= best_threshold).astype(int)
print(classification_report(y_test, best_pred, target_names=['non_landslide', 'landslide']))
cm = confusion_matrix(y_test, best_pred)
ConfusionMatrixDisplay(cm, display_labels=['non_landslide', 'landslide']).plot(cmap='Blues')
plt.title(f'{best_name}: threshold={best_threshold:.2f}')
plt.tight_layout()
plt.show()

               precision    recall  f1-score   support

non_landslide       0.33      0.38      0.35      1138
    landslide       0.37      0.32      0.34      1320

     accuracy                           0.35      2458
    macro avg       0.35      0.35      0.35      2458
 weighted avg       0.35      0.35      0.35      2458



## 8. Calibration

In [16]:
best_oof = oof_proba[best_name]
calibrator = IsotonicRegression(out_of_bounds='clip').fit(best_oof, y_train)
cal_proba = calibrator.predict(best_proba)

print('Raw Brier:', round(brier_score_loss(y_test, best_proba), 4))
print('Calibrated Brier:', round(brier_score_loss(y_test, cal_proba), 4))

plt.figure(figsize=(5.5, 5))
for label, p in [('raw', best_proba), ('calibrated', cal_proba)]:
    frac_pos, mean_pred = calibration_curve(y_test, p, n_bins=10, strategy='quantile')
    plt.plot(mean_pred, frac_pos, marker='o', label=label)
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel('Mean predicted probability'); plt.ylabel('Observed landslide fraction')
plt.title(f'Reliability diagram: {best_name}')
plt.legend()
plt.tight_layout()
plt.show()

Raw Brier: 0.3402
Calibrated Brier: 0.2742


## 9. Feature Importance

In [17]:
perm = permutation_importance(best_model, X_test, y_test, scoring='roc_auc',
                              n_repeats=20, random_state=RANDOM_STATE, n_jobs=-1)
importance = pd.DataFrame({
    'feature': X_test.columns, 'importance_mean': perm.importances_mean, 'importance_std': perm.importances_std,
}).sort_values('importance_mean', ascending=False)
importance

,feature,importance_mean,importance_std
7,accum_30d,0.024315,0.003602
1,burst_3h,0.001389,0.000245
0,burst_1h,-0.000007,0.000227
6,accum_15d,-0.002371,0.001930
11,burst_over_accum1d,-0.002732,0.000159
2,burst_6h,-0.003849,0.002182
3,accum_1d,-0.006400,0.002169
12,antecedent_15_30d,-0.009109,0.002852
8,concentration_1d_over_7d,-0.019728,0.005526
9,concentration_3d_over_30d,-0.028761,0.004335


In [18]:
plt.figure(figsize=(8, 5))
order = importance.sort_values('importance_mean')
sns.barplot(data=order, x='importance_mean', y='feature', color='#4C78A8')
plt.xlabel('Permutation importance: ROC-AUC decrease')
plt.title(f'Feature importance: {best_name}')
plt.tight_layout()
plt.show()

## 10. Save Model Artifacts

Saved in the same style as `models/susceptibility/model_metadata.json` from
the susceptibility notebook, so a prediction/overlay script can load either
the same way.

In [19]:
final_model = clone(fitted_models[best_name]).fit(X, y)
model_path = ARTIFACT_DIR / f'{best_name}_rainfall_trigger_pipeline.joblib'
calibrator_path = ARTIFACT_DIR / f'{best_name}_rainfall_trigger_isotonic_calibrator.joblib'
joblib.dump(final_model, model_path)
joblib.dump(calibrator, calibrator_path)

metadata = {
    'best_model': best_name,
    'selection': 'nested spatial CV (ROC-AUC)',
    'best_params': best_params,
    'threshold_raw_probability': float(best_threshold),
    'feature_columns': feature_cols,
    'spatial_block_deg': BLOCK_DEG,
    'cv_results': cv_results.to_dict(orient='records'),
    'test_results': test_results.to_dict(orient='records'),
    'dataset': {
        'landslide_rows': int((df['landslide'] == 1).sum()),
        'non_landslide_rows': int((df['landslide'] == 0).sum()),
        'non_landslide_sampling': 'paired ring sample: one negative per landslide, 2-10 km (max ~22 km) away, same province, '
                                  '>= 2 km from all landslide records, takes its paired landslide date '
                                  '(scripts/generate_paired_negatives.py) -- NOT a real-world-prevalence sample, '
                                  'and NOT rainfall-matched.',
    },
    'limitations': [
        'Each non-landslide point shares its paired landslide\'s date and region (same weather system, within '
        'about one IMERG cell), so rainfall is nearly identical within a pair by design; this dataset cannot '
        'support a rainfall-amount threshold.',
        'Landslide dates are satellite-image dates of scar-mapping layers, not occurrence dates; ~16 percent '
        'may have day and month swapped (docs/METHODS.md, Known issues 3).',
        'ROC-AUC reported here measures whether small within-pair rainfall differences carry any signal -- it '
        'is not a measure of how well rainfall alone predicts landslides in the real world.',
        'No real-world-prevalence rainfall sample (e.g. random dates/locations regardless of paired '
        'landslide wetness) has been collected; building one is the natural next step for an '
        'operationally calibrated rainfall trigger threshold.',
        'Thresholds/metrics here describe relative model ranking, not an operational alert threshold.',
    ],
}
metadata_path = ARTIFACT_DIR / 'rainfall_trigger_metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2, default=str))
print('Saved:', model_path)
print('Saved:', calibrator_path)
print('Saved:', metadata_path)

Saved: models/rainfall_trigger/random_forest_rainfall_trigger_pipeline.joblib
Saved: models/rainfall_trigger/random_forest_rainfall_trigger_isotonic_calibrator.joblib
Saved: models/rainfall_trigger/rainfall_trigger_metadata.json
